In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
leiden_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(leiden_path, sep='\t', index_col=0)
leiden.shape

In [ ]:
leiden.head()

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
adata_list = []
for d in donors:
    adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/{d}.h5ad')
    adata.obs['rna_cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']
    adata.obs['leiden'] = leiden.reindex(adata.obs.index)['final_leiden']
    adata_list.append(adata)

In [ ]:
for tmp_adata in adata_list:
    sc.pl.umap(tmp_adata, color=['rna_cluster', 'leiden'])

### isolate fail1 traj

In [ ]:
whitelist = ['Start', 'Delayed', 'Fail1']

tmp_list = []
for tmp_adata in adata_list:
    mask = np.array([x in whitelist for x in tmp_adata.obs['rna_cluster']])
    fail_adata = tmp_adata[mask]
    sc.tl.pca(fail_adata)
    sc.pp.neighbors(fail_adata, random_state=0)
    sc.tl.umap(fail_adata, random_state=0)
    sc.pl.umap(fail_adata, color=['rna_cluster', 'leiden'])
    tmp_list.append(fail_adata)

In [ ]:
umap_list = []
for tmp_adata in tmp_list:
    umap_df = pd.DataFrame(index=tmp_adata.obs.index)
    umap_df['umap0'] = tmp_adata.obsm['X_umap'][:,0]
    umap_df['umap1'] = tmp_adata.obsm['X_umap'][:,1]

    select = ['line', 'rna_cluster', 'leiden']
    for s in select:
        umap_df[s] = tmp_adata.obs[s]
    umap_list.append(umap_df)
    
merged = pd.concat(umap_list)
merged.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/mcg_fail1_start_umap.csv', sep='\t')

In [ ]:
!date